# Суу — fine-tune Qwen2.5-1.5B-Instruct (LoRA)

SuuPrep-ийн SAT Math багш "Суу"-г жижиг моделоор fine-tune хийнэ.

**Colab дээр:** Runtime → Change runtime type → **T4 GPU** → Runtime → **Run all**. Нийт ~25–35 минут.

Алхмууд: 1) суулгах · 2) өгөгдөл татах · 3) **base** моделийг үнэлэх · 4) LoRA сургалт · 5) **fine-tuned** моделийг үнэлэх · 6) GGUF гаргаж Ollama-д ашиглах.

In [ ]:
%%capture
!pip install unsloth

In [ ]:
# Data comes from the repo (ai/data/tuning/*.jsonl, built by `bun ai/train/export.ts`).
REPO = "pinecone-studio/pinequest-s5-episode-2-team-12"
BRANCH = "main"   # change to your branch name if the PR is not merged yet
BASE = f"https://raw.githubusercontent.com/{REPO}/{BRANCH}/ai/data/tuning"
for f in ["train.jsonl", "val.jsonl", "eval.jsonl"]:
    !wget -q -O {f} {BASE}/{f}
!wc -l train.jsonl val.jsonl eval.jsonl

In [ ]:
import json
def read_jsonl(path):
    with open(path, encoding="utf-8") as fh:
        return [json.loads(l) for l in fh if l.strip()]
train_rows, val_rows, eval_rows = read_jsonl("train.jsonl"), read_jsonl("val.jsonl"), read_jsonl("eval.jsonl")
SYSTEM_PROMPT = train_rows[0]["messages"][0]["content"]
print(len(train_rows), "train ·", len(val_rows), "val ·", len(eval_rows), "eval")
print(SYSTEM_PROMPT)

In [ ]:
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template
import torch

MAX_LEN = 1024
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-1.5B-Instruct",
    max_seq_length=MAX_LEN,
    load_in_4bit=True,
)
tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")
model = FastLanguageModel.get_peft_model(
    model,
    r=16, lora_alpha=16, lora_dropout=0, bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth", random_state=42,
)

## Үнэлгээ (base vs fine-tuned)
- **Хариу зөв %** — "explain" асуултад зөв эцсийн хариу гарсан эсэх
- **Hint leak %** — "hint" асуултад хариугаа хэлчихсэн эсэх (бага байх тусам сайн)
- **Монгол %** — гаралтын үсгийн хэдэн хувь нь кирилл вэ

In [ ]:
import re

def generate(prompt, max_new_tokens=300):
    msgs = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": prompt}]
    ids = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt").to("cuda")
    out = model.generate(input_ids=ids, max_new_tokens=max_new_tokens, temperature=0.3, do_sample=True)
    return tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True).strip()

def norm(s):
    return re.sub(r"[\s$\\{}]", "", s.lower()).replace("frac", "")

def has_answer(text, accepted, letters=True):
    t = norm(text)
    for a in accepted:
        a = norm(a)
        if not a or (not letters and re.fullmatch(r"[a-e]\)", a)):
            continue
        if re.fullmatch(r"[\d.,/-]+", a):   # plain number: don't match 2 inside 12
            if re.search(r"(?<![\d.])" + re.escape(a) + r"(?![\d])", t): return True
        elif a in t:
            return True
    return False

def mongolian_share(text):
    letters = [c for c in text if c.isalpha()]
    if not letters: return 0.0
    return sum(1 for c in letters if "\u0400" <= c <= "\u04ff") / len(letters)

def evaluate(label):
    FastLanguageModel.for_inference(model)
    rows, correct, n_exp, leaks, n_hint, mn = [], 0, 0, 0, 0, []
    for item in eval_rows:
        out = generate(item["prompt"])
        mn.append(mongolian_share(out))
        if item["mode"] == "explain":
            n_exp += 1; correct += has_answer(out, item["accepted"])
        else:
            n_hint += 1; leaks += has_answer(out, item["accepted"], letters=False)
        rows.append({"id": item["questionId"], "mode": item["mode"], "output": out})
    res = {
        "model": label,
        "answer_correct_%": round(100 * correct / max(n_exp, 1)),
        "hint_leak_%": round(100 * leaks / max(n_hint, 1)),
        "mongolian_%": round(100 * sum(mn) / max(len(mn), 1)),
        "samples": rows,
    }
    print({k: v for k, v in res.items() if k != "samples"})
    return res

base = evaluate("Qwen2.5-1.5B-Instruct (base)")
print("\n--- base sample ---\n", base["samples"][0]["output"][:500])

In [ ]:
from datasets import Dataset
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

def to_text(rows):
    return Dataset.from_list([
        {"text": tokenizer.apply_chat_template(r["messages"], tokenize=False)} for r in rows
    ])

FastLanguageModel.for_training(model)
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=to_text(train_rows),
    eval_dataset=to_text(val_rows),
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=MAX_LEN,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=2,
        num_train_epochs=3,
        learning_rate=2e-4,
        warmup_steps=5,
        lr_scheduler_type="linear",
        logging_steps=5,
        eval_strategy="epoch",
        optim="adamw_8bit",
        weight_decay=0.01,
        seed=42,
        output_dir="outputs",
        report_to="none",
    ),
)
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)
stats = trainer.train()
print(stats)

In [ ]:
tuned = evaluate("Suu (Qwen2.5-1.5B + LoRA)")

lines = ["# Suu eval report", "",
         f"Held-out questions: {len({r['questionId'] for r in eval_rows})} · prompts: {len(eval_rows)} · train examples: {len(train_rows)}", "",
         "| Model | Answer correct % | Hint leak % (lower = better) | Mongolian % |", "|---|---|---|---|"]
for r in (base, tuned):
    lines.append(f"| {r['model']} | {r['answer_correct_%']} | {r['hint_leak_%']} | {r['mongolian_%']} |")
lines += ["", "## Same prompt, before vs after", ""]
for b, t in list(zip(base["samples"], tuned["samples"]))[:4]:
    lines += [f"### {b['id']} · {b['mode']}", "", "**Base:**", "", b["output"], "", "**Suu:**", "", t["output"], ""]
report = "\n".join(lines)
open("eval_report.md", "w", encoding="utf-8").write(report)
from IPython.display import Markdown, display
display(Markdown(report))

## Туршиж үзэх

In [ ]:
FastLanguageModel.for_inference(model)
print(generate("If $2x+3=11$, what is $x$?\n\nЗөвлөгөө өгөөч, хариуг нь битгий хэлээрэй."))

## GGUF гаргах → Ollama

Доорх cell `suu.zip` татна (~1 GB). Laptop дээрээ задлаад:

```bash
cd suu
ollama create suu -f Modelfile
ollama run suu "Сайн уу, Суу!"
```

Дараа нь апп-ийн `.env.local`-д `OLLAMA_URL=http://localhost:11434` ба `OLLAMA_MODEL=suu` гэж бичнэ.

In [ ]:
import glob, os, shutil
model.save_pretrained("suu-lora"); tokenizer.save_pretrained("suu-lora")   # small LoRA adapter (backup)
model.save_pretrained_gguf("suu-gguf", tokenizer, quantization_method="q4_k_m")

os.makedirs("suu", exist_ok=True)
gguf = sorted(glob.glob("suu-gguf*/**/*.gguf", recursive=True) + glob.glob("*.gguf"), key=os.path.getsize)[-1]
shutil.copy(gguf, "suu/suu.gguf")
sys_prompt = SYSTEM_PROMPT.replace('"', '\\"')
modelfile = f'''FROM ./suu.gguf
TEMPLATE """{{{{- if .System }}}}<|im_start|>system
{{{{ .System }}}}<|im_end|>
{{{{ end }}}}{{{{- range .Messages }}}}<|im_start|>{{{{ .Role }}}}
{{{{ .Content }}}}<|im_end|>
{{{{ end }}}}<|im_start|>assistant
"""
SYSTEM "{sys_prompt}"
PARAMETER temperature 0.3
PARAMETER stop "<|im_end|>"
PARAMETER stop "<|im_start|>"
'''
open("suu/Modelfile", "w", encoding="utf-8").write(modelfile)
shutil.copy("eval_report.md", "suu/eval_report.md")
!zip -r -q suu.zip suu
!ls -lh suu suu.zip
from google.colab import files
files.download("suu.zip")